In [1]:
"""
Orbit Prediction & Station-Keeping Analysis
==========================================
Pipeline: TLE Fetch → SGP4 Propagation → Feature Engineering → Dataset Export
Author: Orbit Analysis System
"""

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import math
import warnings
warnings.filterwarnings("ignore")

# ── SGP4 ─────────────────────────────────────────────────────────────────────
try:
    from sgp4.api import Satrec, jday
    SGP4_AVAILABLE = True
except ImportError:
    SGP4_AVAILABLE = False
    print("[WARN] sgp4 not installed. Using simulated TLE data.")

# ── HTTP ──────────────────────────────────────────────────────────────────────
try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False

# ─────────────────────────────────────────────────────────────────────────────
# 1. TLE FETCHING
# ─────────────────────────────────────────────────────────────────────────────

CELESTRAK_URLS = {
    "geo":        "https://celestrak.org/SOCRATES/query.php?CODE=GEO&FORMAT=TLE",
    "active":     "https://celestrak.org/SOCRATES/query.php?CODE=active&FORMAT=TLE",
    "geo_sample": "https://celestrak.org/pub/TLE/geo.txt",
    "stations":   "https://celestrak.org/SOCRATES/query.php?CODE=stations&FORMAT=TLE",
}

# Representative real-world GEO satellite TLEs (INSAT-3D, GSAT-17, etc.)
SAMPLE_TLES = [
    {
        "name": "INSAT-3D",
        "line1": "1 39216U 13042A   24001.50000000  .00000000  00000-0  00000+0 0  9993",
        "line2": "2 39216   0.0305  95.4210 0000945 268.1234 130.5432  1.00273791 39012",
    },
    {
        "name": "GSAT-17",
        "line1": "1 42821U 17041A   24001.50000000  .00000012  00000-0  00000+0 0  9991",
        "line2": "2 42821   0.0412  78.2100 0001234 290.5678  45.3210  1.00273792 23456",
    },
    {
        "name": "GSAT-11",
        "line1": "1 43751U 18086A   24001.50000000  .00000008  00000-0  00000+0 0  9998",
        "line2": "2 43751   0.0189 123.4560 0000812 312.7890  89.1230  1.00273790 19823",
    },
    {
        "name": "SES-12",
        "line1": "1 43488U 18046A   24001.50000000  .00000005  00000-0  00000+0 0  9994",
        "line2": "2 43488   0.0267  45.6789 0001456 178.9012 234.5678  1.00273793 42017",
    },
    {
        "name": "MEASAT-3B",
        "line1": "1 40146U 14058B   24001.50000000  .00000003  00000-0  00000+0 0  9992",
        "line2": "2 40146   0.0334  56.7890 0000678 145.2345 312.7654  1.00273791 34512",
    },
]


def fetch_tle_from_celestrak(group: str = "geo_sample") -> list[dict]:
    """
    Fetch TLEs from CelesTrak. Falls back to sample TLEs if network unavailable.
    Returns list of {'name', 'line1', 'line2'} dicts.
    """
    if not REQUESTS_AVAILABLE:
        print("[INFO] requests not available. Using sample TLE dataset.")
        return SAMPLE_TLES

    url = CELESTRAK_URLS.get(group, CELESTRAK_URLS["geo_sample"])
    print(f"[INFO] Fetching TLEs from: {url}")
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        lines = [l.strip() for l in resp.text.strip().splitlines() if l.strip()]
        tles = []
        for i in range(0, len(lines) - 2, 3):
            if lines[i + 1].startswith("1 ") and lines[i + 2].startswith("2 "):
                tles.append({
                    "name":  lines[i],
                    "line1": lines[i + 1],
                    "line2": lines[i + 2],
                })
        print(f"[INFO] Fetched {len(tles)} TLEs from CelesTrak.")
        return tles if tles else SAMPLE_TLES
    except Exception as e:
        print(f"[WARN] CelesTrak fetch failed ({e}). Using sample TLE dataset.")
        return SAMPLE_TLES


# ─────────────────────────────────────────────────────────────────────────────
# 2. ORBITAL ELEMENT EXTRACTION FROM TLE
# ─────────────────────────────────────────────────────────────────────────────

def parse_tle_elements(tle: dict) -> dict:
    """
    Parse raw Keplerian elements directly from TLE strings.
    Returns dict of classical orbital elements + metadata.
    """
    l1, l2 = tle["line1"], tle["line2"]

    # Line 1
    epoch_year    = int(l1[18:20])
    epoch_day     = float(l1[20:32])
    bstar         = _parse_decimal(l1[53:61])
    mean_motion_d = float(l1[33:43])          # dn/dt (rev/day²)

    # Line 2
    inclination   = float(l2[8:16])           # degrees
    raan          = float(l2[17:25])          # right ascension of ascending node
    eccentricity  = float("0." + l2[26:33])  # dimensionless
    arg_perigee   = float(l2[34:42])          # argument of perigee
    mean_anomaly  = float(l2[43:51])          # degrees
    mean_motion   = float(l2[52:63])          # rev/day

    # Derived
    mu            = 398600.4418               # km³/s²  (Earth's gravitational parameter)
    n_rad_s       = mean_motion * 2 * math.pi / 86400  # rad/s
    semi_major    = (mu / n_rad_s**2) ** (1/3)        # km
    period_min    = 1440.0 / mean_motion               # minutes
    altitude_km   = semi_major - 6371.0                # approx altitude

    # Epoch → datetime
    full_year     = 2000 + epoch_year if epoch_year < 57 else 1900 + epoch_year
    epoch_dt      = datetime(full_year, 1, 1) + timedelta(days=epoch_day - 1)

    return {
        "name":           tle["name"],
        "epoch_dt":       epoch_dt,
        "inclination":    inclination,
        "raan":           raan,
        "eccentricity":   eccentricity,
        "arg_perigee":    arg_perigee,
        "mean_anomaly":   mean_anomaly,
        "mean_motion":    mean_motion,
        "semi_major_km":  round(semi_major, 3),
        "altitude_km":    round(altitude_km, 3),
        "period_min":     round(period_min, 4),
        "bstar":          bstar,
        "mean_motion_dot": mean_motion_d,
    }


def _parse_decimal(s: str) -> float:
    """Parse TLE's packed decimal notation (e.g., '-12345-3' → -0.00012345)."""
    s = s.strip()
    if not s or s in ("00000+0", "00000-0", " 00000-0"):
        return 0.0
    try:
        sign  = -1 if s[0] == "-" else 1
        s     = s.lstrip("+-")
        mant  = s[:-2]
        exp   = int(s[-2:])
        return sign * float("0." + mant) * 10 ** exp
    except Exception:
        return 0.0


# ─────────────────────────────────────────────────────────────────────────────
# 3. SGP4 PROPAGATION
# ─────────────────────────────────────────────────────────────────────────────

def propagate_orbit_sgp4(tle: dict,
                          start: datetime,
                          duration_hours: int = 72,
                          step_minutes: int = 10) -> pd.DataFrame:
    """
    Propagate satellite orbit using SGP4. Returns ECI state vectors + derived params.
    """
    if not SGP4_AVAILABLE:
        return _simulate_orbit(tle, start, duration_hours, step_minutes)

    sat   = Satrec.twoline2rv(tle["line1"], tle["line2"])
    rows  = []
    t     = start
    dt    = timedelta(minutes=step_minutes)

    while t <= start + timedelta(hours=duration_hours):
        yr, mo, dy = t.year, t.month, t.day
        hr, mn, sc = t.hour, t.minute, t.second + t.microsecond / 1e6
        jd, fr     = jday(yr, mo, dy, hr, mn, sc)
        e, r, v    = sat.sgp4(jd, fr)

        if e == 0:  # no error
            rx, ry, rz = r          # km  (ECI)
            vx, vy, vz = v          # km/s
            radius     = math.sqrt(rx**2 + ry**2 + rz**2)
            speed      = math.sqrt(vx**2 + vy**2 + vz**2)
            altitude   = radius - 6371.0

            # Orbital elements from state vector
            h_vec      = np.cross(r, v)          # specific angular momentum
            h_mag      = np.linalg.norm(h_vec)
            inc        = math.degrees(math.acos(
                             np.clip(h_vec[2] / h_mag, -1, 1)))

            rows.append({
                "timestamp":    t,
                "rx_km":        round(rx, 4),
                "ry_km":        round(ry, 4),
                "rz_km":        round(rz, 4),
                "vx_kms":       round(vx, 6),
                "vy_kms":       round(vy, 6),
                "vz_kms":       round(vz, 6),
                "altitude_km":  round(altitude, 4),
                "speed_kms":    round(speed, 6),
                "inclination":  round(inc, 6),
                "satellite":    tle["name"],
            })
        t += dt

    return pd.DataFrame(rows)


def _simulate_orbit(tle: dict,
                     start: datetime,
                     duration_hours: int = 72,
                     step_minutes: int = 10) -> pd.DataFrame:
    """
    Simulated SGP4-like propagation for environments without the sgp4 library.
    Models GEO satellite with realistic perturbations (J2, SRP, lunar/solar).
    """
    elems  = parse_tle_elements(tle)
    rows   = []
    t      = start
    dt_min = step_minutes
    steps  = int(duration_hours * 60 / dt_min)

    # Base orbital parameters
    a      = elems["semi_major_km"]    # km
    e0     = elems["eccentricity"]
    i0     = elems["inclination"]      # degrees
    raan0  = elems["raan"]
    om0    = elems["arg_perigee"]
    M0     = elems["mean_anomaly"]
    n      = elems["mean_motion"]      # rev/day → deg/min
    n_deg  = n * 360 / 1440.0

    # J2 perturbation rates (deg/day for GEO)
    J2       = 1.08262668e-3
    Re       = 6378.137
    n_rad    = n * 2 * math.pi / 1440.0  # rad/min
    J2_factor = -1.5 * J2 * (Re / a)**2 * n_rad
    i_rad    = math.radians(i0)
    raan_dot = J2_factor * math.cos(i_rad) * (180 / math.pi)     # deg/min
    om_dot   = J2_factor * (2 - 2.5 * math.sin(i_rad)**2) * (180 / math.pi)

    mu = 398600.4418
    rng = np.random.default_rng(42)

    for step in range(steps):
        t_min = step * dt_min
        M     = (M0 + n_deg * t_min) % 360
        raan  = (raan0 + raan_dot * t_min) % 360
        om    = (om0   + om_dot   * t_min) % 360

        # Inclination drift: slow secular + sinusoidal (lunar/solar)
        i_drift  = 0.00000012 * t_min          # ~0.75°/yr
        i_lunisol = 0.002 * math.sin(2 * math.pi * t_min / (365.25 * 1440))
        inc      = i0 + i_drift + i_lunisol

        # Eccentricity: SRP-induced variation
        ecc = e0 + 0.0000002 * math.sin(2 * math.pi * t_min / 43200)

        # Position in orbital plane
        E     = _eccentric_anomaly(math.radians(M), ecc)
        nu    = 2 * math.atan2(math.sqrt(1 + ecc) * math.sin(E / 2),
                               math.sqrt(1 - ecc) * math.cos(E / 2))
        r_mag = a * (1 - ecc * math.cos(E))

        # ECI conversion
        om_r  = math.radians(om)
        raan_r= math.radians(raan)
        i_r   = math.radians(inc)

        x_orb = r_mag * math.cos(nu)
        y_orb = r_mag * math.sin(nu)

        rx = ((math.cos(raan_r) * math.cos(om_r) - math.sin(raan_r) * math.sin(om_r) * math.cos(i_r)) * x_orb
            + (-math.cos(raan_r) * math.sin(om_r) - math.sin(raan_r) * math.cos(om_r) * math.cos(i_r)) * y_orb)
        ry = ((math.sin(raan_r) * math.cos(om_r) + math.cos(raan_r) * math.sin(om_r) * math.cos(i_r)) * x_orb
            + (-math.sin(raan_r) * math.sin(om_r) + math.cos(raan_r) * math.cos(om_r) * math.cos(i_r)) * y_orb)
        rz = (math.sin(om_r + nu) * math.sin(i_r) * r_mag)

        # Velocity (vis-viva)
        speed   = math.sqrt(mu * (2 / r_mag - 1 / a))
        altitude= r_mag - 6371.0

        # Small measurement noise (telemetry realism)
        noise = rng.normal(0, 1e-4)

        rows.append({
            "timestamp":    t + timedelta(minutes=t_min),
            "rx_km":        round(rx + noise, 4),
            "ry_km":        round(ry + noise, 4),
            "rz_km":        round(rz + noise, 4),
            "vx_kms":       round(speed * 0.707, 6),
            "vy_kms":       round(speed * 0.707, 6),
            "vz_kms":       round(speed * 0.001, 6),
            "altitude_km":  round(altitude, 4),
            "speed_kms":    round(speed, 6),
            "inclination":  round(inc + noise * 0.01, 6),
            "satellite":    tle["name"],
        })

    return pd.DataFrame(rows)


def _eccentric_anomaly(M: float, e: float, tol: float = 1e-10) -> float:
    """Solve Kepler's equation M = E - e·sin(E) via Newton-Raphson."""
    E = M
    for _ in range(100):
        dE = (M - E + e * math.sin(E)) / (1 - e * math.cos(E))
        E += dE
        if abs(dE) < tol:
            break
    return E


# ─────────────────────────────────────────────────────────────────────────────
# 4. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build ML-ready features from raw propagated state vectors.
    Adds time-domain, kinematic, and perturbation-proxy features.
    """
    df = df.copy().sort_values("timestamp").reset_index(drop=True)

    # Time features
    df["t_hours"]   = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds() / 3600
    df["day_of_year"] = df["timestamp"].dt.dayofyear
    df["hour_of_day"] = df["timestamp"].dt.hour + df["timestamp"].dt.minute / 60

    # Cyclical encoding (avoids discontinuities at 0/360)
    df["sin_hour"]  = np.sin(2 * np.pi * df["hour_of_day"] / 24)
    df["cos_hour"]  = np.cos(2 * np.pi * df["hour_of_day"] / 24)
    df["sin_doy"]   = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
    df["cos_doy"]   = np.cos(2 * np.pi * df["day_of_year"] / 365.25)

    # Kinematics
    df["r_mag_km"]  = np.sqrt(df["rx_km"]**2 + df["ry_km"]**2 + df["rz_km"]**2)
    df["v_mag_kms"] = np.sqrt(df["vx_kms"]**2 + df["vy_kms"]**2 + df["vz_kms"]**2)
    df["lat_proxy"] = np.degrees(np.arcsin(np.clip(df["rz_km"] / df["r_mag_km"], -1, 1)))
    df["lon_proxy"] = np.degrees(np.arctan2(df["ry_km"], df["rx_km"]))

    # Specific orbital energy (vis-viva)
    mu = 398600.4418
    df["sp_energy"]  = df["v_mag_kms"]**2 / 2 - mu / df["r_mag_km"]
    df["semi_major"] = -mu / (2 * df["sp_energy"])

    # Lagged features (drift indicators)
    for lag in [1, 3, 6, 12]:
        df[f"inc_lag{lag}"]  = df["inclination"].shift(lag)
        df[f"alt_lag{lag}"]  = df["altitude_km"].shift(lag)

    # Rolling statistics (station-keeping telemetry pattern)
    df["inc_roll6_mean"]  = df["inclination"].rolling(6).mean()
    df["inc_roll6_std"]   = df["inclination"].rolling(6).std()
    df["alt_roll6_mean"]  = df["altitude_km"].rolling(6).mean()
    df["inc_delta"]       = df["inclination"].diff()        # first difference (drift rate)
    df["alt_delta"]       = df["altitude_km"].diff()

    # Station-keeping flag: |inclination| > 0.05° triggers N/S manoeuvre
    df["ns_manoeuvre_needed"] = (df["inclination"].abs() > 0.05).astype(int)
    df["ew_manoeuvre_needed"] = (df["alt_delta"].abs() > 0.5).astype(int)

    return df.dropna().reset_index(drop=True)


# ─────────────────────────────────────────────────────────────────────────────
# 5. PIPELINE ORCHESTRATOR
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(tles: list[dict] = None,
                 duration_hours: int = 120,
                 step_minutes: int = 10,
                 start: datetime = None) -> pd.DataFrame:
    """
    Full pipeline: TLE list → propagation → feature engineering → combined DataFrame.
    """
    if tles is None:
        tles = SAMPLE_TLES
    if start is None:
        start = datetime(2026, 2, 1, 0, 0, 0)

    all_frames = []
    for tle in tles:
        print(f"  → Propagating: {tle['name']}")
        raw = propagate_orbit_sgp4(tle, start, duration_hours, step_minutes)
        fe  = engineer_features(raw)
        all_frames.append(fe)

    combined = pd.concat(all_frames, ignore_index=True)
    print(f"[INFO] Pipeline complete. Dataset: {len(combined):,} rows × {len(combined.columns)} cols")
    return combined


# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print(" Orbit Prediction Pipeline")
    print("=" * 60)

    tles = fetch_tle_from_celestrak("geo_sample")
    tles = tles[:5]  # use first 5

    df = run_pipeline(tles, duration_hours=120, step_minutes=10)

    out = "orbit_dataset.csv"
    df.to_csv(out, index=False)
    print(f"\n[DONE] Saved → {out}")
    print(df[["timestamp", "satellite", "inclination", "altitude_km",
              "ns_manoeuvre_needed"]].head(10).to_string(index=False))

[WARN] sgp4 not installed. Using simulated TLE data.
 Orbit Prediction Pipeline
[INFO] Fetching TLEs from: https://celestrak.org/pub/TLE/geo.txt
[WARN] CelesTrak fetch failed (403 Client Error: Forbidden for url: https://celestrak.org/pub/TLE/geo.txt). Using sample TLE dataset.
  → Propagating: INSAT-3D
  → Propagating: GSAT-17
  → Propagating: GSAT-11
  → Propagating: SES-12
  → Propagating: MEASAT-3B
[INFO] Pipeline complete. Dataset: 3,540 rows × 39 cols

[DONE] Saved → orbit_dataset.csv
          timestamp satellite  inclination  altitude_km  ns_manoeuvre_needed
2026-02-01 02:00:00  INSAT-3D     0.030517   35796.9290                    0
2026-02-01 02:10:00  INSAT-3D     0.030520   35796.9833                    0
2026-02-01 02:20:00  INSAT-3D     0.030521   35797.0302                    0
2026-02-01 02:30:00  INSAT-3D     0.030521   35797.0697                    0
2026-02-01 02:40:00  INSAT-3D     0.030523   35797.1018                    0
2026-02-01 02:50:00  INSAT-3D     0.030524

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# MAIN (Corrected for Notebook / Colab)
# Assumes code1 cell already executed
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    print("=" * 60)
    print(" Orbit Prediction — Model Training & Evaluation")
    print("=" * 60)

    # Uses functions already loaded from previous cell
    df = run_pipeline(SAMPLE_TLES[:3], duration_hours=120)

    # Train per-satellite and combined
    for sat_name in df["satellite"].unique():

        print(f"\n── {sat_name} ──")

        inc_data, alt_data = prepare_datasets(
            df,
            satellite=sat_name,
            horizon_steps=6
        )

        if len(inc_data["X_train"]) < 20:
            print("  [SKIP] Not enough data.")
            continue

        predictor = OrbitPredictor(
            model_type="gradient_boosting"
        )

        predictor.fit(inc_data, alt_data)

        metrics = predictor.evaluate(
            inc_data,
            alt_data
        )

        print(
            f"  Inclination → "
            f"MAE: {metrics['inclination']['MAE']:.6f}°   "
            f"RMSE: {metrics['inclination']['RMSE']:.6f}°"
        )

        print(
            f"  Altitude    → "
            f"MAE: {metrics['altitude']['MAE']:.4f} km   "
            f"RMSE: {metrics['altitude']['RMSE']:.4f} km"
        )

        # Delta-V estimate
        mean_inc = df[
            df["satellite"] == sat_name
        ]["inclination"].abs().mean()

        dv = estimate_delta_v(mean_inc)

        print(
            f"  ΔV_NS (avg): {dv['delta_v_NS_ms']:.4f} m/s   "
            f"| Annual budget: {dv['annual_budget_ms']:.2f} m/s"
        )

    # ─────────────────────────────────────────
    # Time-Series Forecast Demo
    # ─────────────────────────────────────────
    print("\n── Time-Series Forecast (AR model) ──")

    sat_df = df[
        df["satellite"] == df["satellite"].iloc[0]
    ].sort_values("t_hours")

    inc_series = sat_df["inclination"].values

    ts = TimeSeriesForecaster(p=12)

    ts.fit(inc_series[:-72])

    forecast = ts.forecast(
        inc_series[:-72],
        steps=72
    )

    lo, hi = ts.confidence_band(
        steps=72
    )

    mae_ts = np.mean(
        np.abs(inc_series[-72:] - forecast)
    )

    rmse_ts = np.sqrt(
        np.mean((inc_series[-72:] - forecast) ** 2)
    )

    print(
        f"  AR(12) Forecast → "
        f"MAE: {mae_ts:.6f}°   "
        f"RMSE: {rmse_ts:.6f}°"
    )

    print(
        f"  95% CI at step 72: ±{hi[-1]:.6f}°"
    )

 Orbit Prediction — Model Training & Evaluation
  → Propagating: INSAT-3D
  → Propagating: GSAT-17
  → Propagating: GSAT-11
[INFO] Pipeline complete. Dataset: 2,124 rows × 39 cols

── INSAT-3D ──
[MODEL] Training gradient_boosting models …
[MODEL] Training complete.
  Inclination → MAE: 0.000119°   RMSE: 0.000135°
  Altitude    → MAE: 0.0220 km   RMSE: 0.0305 km
  ΔV_NS (avg): 1.6651 m/s   | Annual budget: 5.33 m/s

── GSAT-17 ──
[MODEL] Training gradient_boosting models …
[MODEL] Training complete.
  Inclination → MAE: 0.000128°   RMSE: 0.000139°
  Altitude    → MAE: 0.0252 km   RMSE: 0.0312 km
  ΔV_NS (avg): 2.2394 m/s   | Annual budget: 6.48 m/s

── GSAT-11 ──
[MODEL] Training gradient_boosting models …
[MODEL] Training complete.
  Inclination → MAE: 0.000123°   RMSE: 0.000136°
  Altitude    → MAE: 0.0210 km   RMSE: 0.0278 km
  ΔV_NS (avg): 1.0426 m/s   | Annual budget: 4.09 m/s

── Time-Series Forecast (AR model) ──
  AR(12) Forecast → MAE: 0.000001°   RMSE: 0.000002°
  95% CI at s

In [6]:
# ===============================================================
# Orbit Prediction — Full Evaluation (Corrected for Notebook/Colab)
# Run code1 cell first, then code2/model cell first
# No external module imports required
# ===============================================================

import numpy as np
import pandas as pd
import json
from datetime import datetime

# ---------------------------------------------------------------
# CROSS VALIDATION
# ---------------------------------------------------------------

def walk_forward_cv(df, satellite, n_splits=5, model_type="gradient_boosting"):

    sat_df = df[df["satellite"] == satellite].sort_values("t_hours").reset_index(drop=True)

    avail = [c for c in FEATURE_COLS if c in sat_df.columns]

    n = len(sat_df)
    fold_size = n // (n_splits + 1)

    mae_scores = []
    rmse_scores = []

    for fold in range(n_splits):

        train_end = fold_size * (fold + 1)
        test_end = min(train_end + fold_size, n)

        if train_end < 20 or test_end <= train_end:
            continue

        train_df = sat_df.iloc[:train_end]
        test_df = sat_df.iloc[train_end:test_end]

        X_train = train_df[avail].values
        y_train = train_df["inclination"].values

        X_test = test_df[avail].values
        y_test = test_df["inclination"].values

        if len(X_train) < 10 or len(X_test) < 3:
            continue

        try:
            from sklearn.ensemble import GradientBoostingRegressor
            from sklearn.preprocessing import StandardScaler

            scaler = StandardScaler()

            Xtr = scaler.fit_transform(X_train)
            Xte = scaler.transform(X_test)

            model = GradientBoostingRegressor(
                n_estimators=100,
                learning_rate=0.05,
                max_depth=3,
                subsample=0.8,
                random_state=fold
            )

            model.fit(Xtr, y_train)
            preds = model.predict(Xte)

        except:
            model = _NumpyRidgeRegression()
            model.fit(X_train, y_train)
            preds = model.predict(X_test)

        mae = np.mean(np.abs(y_test - preds))
        rmse = np.sqrt(np.mean((y_test - preds) ** 2))

        mae_scores.append(mae)
        rmse_scores.append(rmse)

    return {
        "satellite": satellite,
        "folds": len(mae_scores),
        "MAE_mean": round(float(np.mean(mae_scores)), 6),
        "RMSE_mean": round(float(np.mean(rmse_scores)), 6)
    }


# ---------------------------------------------------------------
# RESIDUAL ANALYSIS
# ---------------------------------------------------------------

def residual_analysis(residual_df):

    r = residual_df["residual"].values

    return {
        "mean": round(float(np.mean(r)), 8),
        "std": round(float(np.std(r)), 8),
        "max_abs": round(float(np.max(np.abs(r))), 8),
        "p95": round(float(np.percentile(np.abs(r), 95)), 8)
    }


# ---------------------------------------------------------------
# STATION KEEPING SUMMARY
# ---------------------------------------------------------------

def station_keeping_summary(df):

    rows = []

    for sat in df["satellite"].unique():

        sdf = df[df["satellite"] == sat]

        mean_inc = sdf["inclination"].abs().mean()
        max_inc = sdf["inclination"].abs().max()

        dv = estimate_delta_v(mean_inc)

        rows.append({
            "satellite": sat,
            "mean_inc_deg": round(mean_inc, 6),
            "max_inc_deg": round(max_inc, 6),
            "dv_NS_ms": dv["delta_v_NS_ms"],
            "annual_budget_ms": dv["annual_budget_ms"]
        })

    return pd.DataFrame(rows)


# ---------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------

def main():

    print("=" * 65)
    print(" Orbit Prediction & Station-Keeping Evaluation")
    print("=" * 65)

    print("\n[1] Generating Dataset...\n")

    df = run_pipeline(
        SAMPLE_TLES[:5],
        duration_hours=120,
        step_minutes=10
    )

    print(f"Rows: {len(df)}")
    print(f"Satellites: {df['satellite'].nunique()}")

    print("\n[2] Model Training...\n")

    all_metrics = []
    predictors = {}
    datasets = {}

    for sat in df["satellite"].unique():

        print(f"--- {sat} ---")

        inc_data, alt_data = prepare_datasets(
            df,
            satellite=sat,
            horizon_steps=6
        )

        if len(inc_data["X_train"]) < 30:
            print("Skipped")
            continue

        model = OrbitPredictor(
            model_type="gradient_boosting"
        )

        model.fit(inc_data, alt_data)

        metrics = model.evaluate(
            inc_data,
            alt_data
        )

        predictors[sat] = model
        datasets[sat] = inc_data

        print("Inclination MAE :", metrics["inclination"]["MAE"])
        print("Inclination RMSE:", metrics["inclination"]["RMSE"])
        print("Altitude MAE    :", metrics["altitude"]["MAE"])
        print("Altitude RMSE   :", metrics["altitude"]["RMSE"])
        print()

        all_metrics.append({
            "satellite": sat,
            "inc_MAE": metrics["inclination"]["MAE"],
            "inc_RMSE": metrics["inclination"]["RMSE"],
            "alt_MAE": metrics["altitude"]["MAE"],
            "alt_RMSE": metrics["altitude"]["RMSE"]
        })

    print("[3] Cross Validation...\n")

    for sat in predictors.keys():

        cv = walk_forward_cv(df, sat)

        print(
            sat,
            "MAE:",
            cv["MAE_mean"],
            "RMSE:",
            cv["RMSE_mean"]
        )

    print("\n[4] Residual Analysis...\n")

    for sat in predictors.keys():

        res = predictors[sat].residuals(
            datasets[sat]
        )

        ra = residual_analysis(res)

        print(
            sat,
            "STD:",
            ra["std"],
            "P95:",
            ra["p95"]
        )

    print("\n[5] Station Keeping Summary...\n")

    sk_df = station_keeping_summary(df)

    print(sk_df.to_string(index=False))

    print("\n[6] Time Series Forecast...\n")

    first_sat = df["satellite"].iloc[0]

    sat_df = df[df["satellite"] == first_sat].sort_values("t_hours")

    series = sat_df["inclination"].values

    ts = TimeSeriesForecaster(p=12)

    ts.fit(series[:-72])

    forecast = ts.forecast(series[:-72], steps=72)

    mae = np.mean(np.abs(series[-72:] - forecast))
    rmse = np.sqrt(np.mean((series[-72:] - forecast) ** 2))

    print("Satellite:", first_sat)
    print("AR Forecast MAE :", round(mae, 6))
    print("AR Forecast RMSE:", round(rmse, 6))

    print("\n[7] Saving Files...\n")

    metrics_df = pd.DataFrame(all_metrics)

    metrics_df.to_csv("eval_metrics.csv", index=False)
    sk_df.to_csv("station_keeping_budget.csv", index=False)
    df.to_csv("orbit_dataset.csv", index=False)

    summary = {
        "rows": int(len(df)),
        "satellites": int(df["satellite"].nunique()),
        "best_MAE": float(metrics_df["inc_MAE"].min())
    }

    with open("summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("Saved:")
    print("eval_metrics.csv")
    print("station_keeping_budget.csv")
    print("orbit_dataset.csv")
    print("summary.json")

    print("\nDONE")


# ---------------------------------------------------------------
# RUN
# ---------------------------------------------------------------

main()

 Orbit Prediction & Station-Keeping Evaluation

[1] Generating Dataset...

  → Propagating: INSAT-3D
  → Propagating: GSAT-17
  → Propagating: GSAT-11
  → Propagating: SES-12
  → Propagating: MEASAT-3B
[INFO] Pipeline complete. Dataset: 3,540 rows × 39 cols
Rows: 3540
Satellites: 5

[2] Model Training...

--- INSAT-3D ---
[MODEL] Training gradient_boosting models …
[MODEL] Training complete.
Inclination MAE : 0.000119
Inclination RMSE: 0.000135
Altitude MAE    : 0.022
Altitude RMSE   : 0.0305

--- GSAT-17 ---
[MODEL] Training gradient_boosting models …
[MODEL] Training complete.
Inclination MAE : 0.000128
Inclination RMSE: 0.000139
Altitude MAE    : 0.0252
Altitude RMSE   : 0.0312

--- GSAT-11 ---
[MODEL] Training gradient_boosting models …
[MODEL] Training complete.
Inclination MAE : 0.000123
Inclination RMSE: 0.000136
Altitude MAE    : 0.021
Altitude RMSE   : 0.0278

--- SES-12 ---
[MODEL] Training gradient_boosting models …
[MODEL] Training complete.
Inclination MAE : 0.000128
Incli